# Welcome to our city housing exploration

This is a hands-on tour of one question — *is our city building the homes it promised?* — answered not from a press release but from the city's own published data, one checkable step at a time.

It's written for a **curious beginner: you don't need to know how to code to start.** If you can open a notebook and run it top to bottom, you can follow every step here — and by the end you'll do something most people never get to: take a real number you computed yourself and set it beside the city's own report to the state, to see whether they agree.

![Berkeley housing, mapped from open city data](https://raw.githubusercontent.com/blockXblock/berkeley-housing-analysis/main/notebooks/curriculum/assets/berkeley_maplibre.jpg)

*By the end of this course, you'll build this — a 3D map of Berkeley's housing project pipeline in June, 2026, made entirely from the city's open data. You'll make it interactive in JN0e.*

The basic data you'll examine came from two California Public Records Act requests to the City of Berkeley for all residential building permits issued by the City between January 1, 2018 and December 31, 2025. It's free. It's easy to do. As you'll see, it's not so easy to turn 32,202 permit documents covering everything from new sewer laterals or signs to a new 34-story downtown apartment building into a clear picture of what housing we built in Berkeley. And Berkeley, Oakland, and San Francisco use a private data company named Tyler Technologies, in Texas, who recently bought the older data company, Accela, with the old city contracts to manage permits. Their open data access is very difficult to use. Berkeley has fired them, as has San Francisco, which makes open data access for major cities uncertain. We hope Berkeley city experts will prevail in making our public data easily accessible.

## How to run this computational notebook

There are two ways to run these computational notebooks, and **either one works** — the notebook adapts to wherever it finds itself:

- **In the cloud (easiest):** open it in **Google Colab** — nothing to install, nothing to download. Click and go.
- **On your own machine:** run it in **Jupyter** or **Jupyter Lab** locally, if you'd rather keep your own copy and tinker.

Either way, the notebook **pulls in Berkeley's open data automatically** as its worked examples — you never have to hunt down or download a spreadsheet yourself. The first code cell sets that up; just run it.

*Coming soon — more cities.* San Francisco publishes their data through an open **API** (a standing web service you can query directly), so they're easy to add — we'll bring them in soon. Berkeley currently doesn't offer one, which is why we work from its permit **spreadsheet exports** instead. Oakland is easier than Berkeley.  Same destination, slightly different on-ramp.

## Data Science classes and Computational Notebooks began in Berkeley
A combined project of the UCB departments of Computer Science and Statistics resulted in over 30,000 Berkeley students taking Data Science 8 (https://data8.org) over the past 8 years, the creation of the new College of Computing, Data Science and Society (https://cdss.berkeley.edu/)run by Jennifer Chayes, and the creation by Fernando Pérez of Jupyter computational notebooks, the open version of Stephen Wolfram's Mathematica notebooks. Here's the classic introduction by Ani Adhikari, John DeNero, and David Wagner: (https://inferentialthinking.com/)

### Running the cells in a Jupyter notebook

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell if you're on an iPad. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.  

To really go fast, go to the top of the page of a notebok,  click **Run All**, and watch what happens.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN00 · Look at the data first](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN00_look_first.ipynb)  |  Next: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb) →

# JN0a · Why municipal data

*On-ramp 1 of 8 — read before JN1.*

Berkeley made a promise to the State of California: over eight years, make room for about **9,000 new homes**. Did it happen? Who actually checks — and how would *you*, personally, find out?

That question is why this course exists. The answer is buried in data the city already publishes — but *published* and *checkable* are not the same thing. This notebook is about why that gap matters, and why, for the first time, someone who has never written a line of code can close it.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout so the cells below find everything.

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## What's a repo?

The code below "walks up" the folders to find the **repo root** — so before it runs, it's worth knowing what a repo is.

A **repository** — "repo" for short — is the project's folder of files, but with a superpower: it's kept under *version control*, a powerful tool that records **every change and update** over the life of the project. Nothing is quietly lost — you can see what changed, when, and why, compare any two moments, and roll back if something goes wrong. That tracked history is what lets a project like this one stay trustworthy. This whole course lives in one repo.

A couple of terms you'll meet right away:

- **Repo root** — the single top folder that everything else sits inside. The notebooks find the data by their *position relative to* this root, so they keep working no matter where you put the project. (That's exactly what the next code cell does.)
- **Clone** — your own local copy of the repo, made when you want to run things on your own machine, tinker, or keep a personal copy. You **don't** need to clone to start: Colab can run a notebook for you without one. Clone when you want to settle in; skip it when you just want to look.

*First-time setup, gently.* If you do keep a local copy, put the project somewhere tidy and easy to find — a dedicated folder you'll remember, not buried among random downloads. A clean root matters because everything here is located *relative to* that top folder: keep it intact and the notebooks will always find their data. We stay deliberately light on commands here — later, **JN0g** and **JN0h** show how an AI agent actually works *inside* a repo like this one.

## Point the notebook at the data

Finds the repo root, locates the permit feed, and puts the project's real shared code on the path. The two knobs or changeable links near the top are all a student changes to run this on another city.

In [ ]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


### What just happened?
We just pulled the two large permit spreadsheet files the City gave us across the network into our folder,or repo root, along with some helper functions.

## What is "open data," and why should you care?

**Open data** is information a government publishes for anyone to use — often because the law (a *public records* or *CPRA* request, in California) requires it. A **building permit** is one such record: the city's permission slip to build, renovate, or demolish. Every new home in Berkeley starts as one of these rows.

So the raw material is *right there*. Let's open it and see what "right there" actually looks like.

In [ ]:
import pandas as pd, glob
_f = sorted(glob.glob(PERMIT_GLOB))[0]            # take the first permit spreadsheet in the feed
raw = pd.read_excel(_f, header=None, dtype=str)   # read it with NO assumed header — exactly as the city shipped it
print('the raw file is', raw.shape[0], 'rows x', raw.shape[1], 'columns — here is the very top:')
raw.head(8)                                        # peek at the top rows (title + blanks sit above the real columns)

### What just happened?
The spreadsheet files were converted into a more stable file format to allow powerful analytic programs, called "pandas", shortened to "pd", to show you the data. It only shows the first 8 rows, but if you change the "8" in the raw.head(8) function, to any number you want, it will show you that many rows when you re-run the cell. 

In [ ]:
# find which of the top rows actually holds the column names
_hdr = next(i for i in range(9) if 'PermitNumber' in raw.iloc[i].astype(str).tolist())
from IPython.display import display
display(raw.head(_hdr + 3))   # the junk rows on top + the header row + a couple of data rows
md(f'''### A pile of data is not an answer

Read as we received it from the City, the top of the file is mostly junk — a title line and blank rows. Look at **row {_hdr}** in the table above (counting from 0): that's the first row where the real column names (`PermitNumber`, `Submittal Date`, …) finally appear — everything above it is title and blanks. The city *published* this — but published is not the same as **legible**: *can an ordinary person get a defensible answer out of it?* Not yet. Making it legible is **JN1**'s job; here we're just looking.''')

### How did we find the header?

We didn't *guess* that the column names sit on a particular row — we **searched** for them. The code scanned down the first several rows looking for the one that contains `PermitNumber`, and used that row as the header. So if the City ever ships the file with a different number of title rows on top, the notebook still finds the real header on its own instead of quietly breaking.

In [ ]:
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)   # stack every yearly file into one table
df = df[df['PermitNumber'].notna()].copy()        # drop rows that have no permit number
# count the rows, the distinct permits, and how many are tagged New construction
n_rows = len(df); n_unique = df['PermitNumber'].nunique(); n_new = int((df['Work Type'] == 'New').sum())
n_other = n_unique - n_new                          # everything else: alterations, additions, demolitions, signs
from IPython.display import display
print(f'the full feed — both files stacked — is {df.shape[0]:,} rows x {df.shape[1]} columns')   # shape of the loaded table
display(df.head(10))                                # show the first 10 rows of the clean, combined table
md(f'''### Two numbers that already complicate the question

Loaded as a real table, the **full feed — both yearly files stacked together** — holds **{n_rows:,}** rows (more than the **{raw.shape[0]:,}** rows in the single file we peeked at above, because that one was just *one* of the two exports). Those are **{n_unique:,}** distinct permits across 2018–2025. But only **{n_new:,}** are tagged **New** construction. The other ~**{n_other:,}** are alterations, additions, demolitions, even signs — real city activity, but not new homes. So even *how many permits?* immediately becomes *how many of which kind?* — and learning to ask that precisely is the whole course.''')

## The arc this series walks

From this messy feed we'll build, step by step: raw permits → a clean table → **buildings** (many permits describe one building) → **completed homes** → and finally a number we can set beside *the city's own report to the state* and check. Each notebook is one honest step, and every step shows its work.

Every year, on April 1, the State Housing and Community Development Agency requires every city in the State of California to submit an Annual Performance Report for the prior calendar year. What did you actually build? Who can afford it? Who got building permits to build new housing? Who applied for permission to get building permits?

The APR lists every application for projects to build new housing units, with the number of units; lists every building permit issued to projects after a city approves, or entitles each project; and, finally, lists every completed housing unit(CO) in that year. The CO, or Certificate of Occupancy, is listed by project, and grouped by the rent level charged, by income level: Acutely Low Income, Extremely Low Income, Very Low Income, Moderate Income, Above Moderate Income.

Our challenge: how to map our building permits to these complicated categories? And once we do, how can we compare our answer to the City's answer? The City's answer is published at the State HCD website; we'll touch that site to compare.

## Why *you* can do this now

Two things changed. **Computational Notebooks** (next) let prose and live code sit together, so every number carries its proof. And **AI agents** (JN0g–JN0h) let you direct that code in plain English. You don't need to become a programmer — you need to learn to ask precise questions and check the answers. That's a skill, not a degree.

**Next — JN0b:** the notebook itself — what it is, and why re-running it *is* re-checking it.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN00 · Look at the data first](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN00_look_first.ipynb)  |  Next: [JN0b · What a computational notebook is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0b_notebook.ipynb) →